In [1]:


import json
import time
import requests


In [2]:

CIUDAD = "València"

AMENITIES = [
    "parking",
    "bicycle_parking",
    "amenity=charging_station",
]

# Endpoint de Overpass API (hay varios espejos por si uno falla)
OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter",
]
 
# Reintentos por endpoint antes de pasar al siguiente
MAX_REINTENTOS_POR_ENDPOINT = 3
 
# Espera base (segundos) para el backoff exponencial: 1º intento espera ESPERA_BASE,
# 2º intento espera ESPERA_BASE*2, 3º intento ESPERA_BASE*4, etc.
ESPERA_BASE = 15
 
# Si nos da 429, esperamos más tiempo todavía (Overpass suele tardar en liberar el slot)
ESPERA_TRAS_429 = 30


SALIDA_GEOJSON = "/Users/aidavillalba/Desktop/descargas_multimodalpy/osm_amenities_valencia.geojson"

In [3]:
def construir_query(ciudad: str, amenities: list[str]) -> str:
    """Construye la query Overpass QL para un área y una lista de amenities."""
    filtros = "".join(
        f'  node["amenity"="{a}"](area.searchArea);\n'
        f'  way["amenity"="{a}"](area.searchArea);\n'
        f'  relation["amenity"="{a}"](area.searchArea);\n'
        for a in amenities
    )
    query = f"""
    [out:json][timeout:180];
    area["name"="{ciudad}"]["boundary"="administrative"]->.searchArea;
    (
    {filtros}
    );
    out center tags;
    """
    return query
 
 
def descargar_datos(query: str) -> dict:
    """
    Envía la query a Overpass API. Para cada endpoint hace varios reintentos con
    backoff exponencial, y si recibe un 429 (Too Many Requests) espera un tiempo
    fijo más largo antes de reintentar, ya que Overpass limita peticiones por IP.
    """
    ultimo_error = None
 
    for endpoint in OVERPASS_ENDPOINTS:
        for intento in range(1, MAX_REINTENTOS_POR_ENDPOINT + 1):
            try:
                print(f"Consultando {endpoint} (intento {intento}/{MAX_REINTENTOS_POR_ENDPOINT}) ...")
                resp = requests.post(endpoint, data={"data": query}, timeout=180)
 
                if resp.status_code == 429:
                    print(f"  429 Too Many Requests. Esperando {ESPERA_TRAS_429}s antes de reintentar...")
                    time.sleep(ESPERA_TRAS_429)
                    ultimo_error = "429 Too Many Requests"
                    continue
 
                # Overpass devuelve 504 cuando el servidor está saturado (no es error nuestro)
                if resp.status_code == 504:
                    espera = ESPERA_BASE * (2 ** (intento - 1))
                    print(f"  504 Gateway Timeout (servidor saturado). Esperando {espera}s...")
                    time.sleep(espera)
                    ultimo_error = "504 Gateway Timeout"
                    continue
 
                resp.raise_for_status()
                return resp.json()
 
            except requests.exceptions.RequestException as e:
                espera = ESPERA_BASE * (2 ** (intento - 1))
                print(f"  Fallo con {endpoint}: {e}. Esperando {espera}s antes de reintentar...")
                ultimo_error = e
                time.sleep(espera)
 
        print(f"Se agotaron los reintentos con {endpoint}, probando el siguiente endpoint...\n")
 
    raise RuntimeError(
        f"No se pudo consultar Overpass API tras probar todos los endpoints. "
        f"Último error: {ultimo_error}\n"
        f"Sugerencia: espera unos minutos y vuelve a ejecutar el script, "
        f"o reduce el área/número de amenities para que la query sea más ligera."
    )
 
 
def a_geojson(data: dict) -> dict:
    """Convierte la respuesta de Overpass (formato JSON) a GeoJSON estándar."""
    features = []
 
    for elemento in data.get("elements", []):
        tipo = elemento.get("type")
 
        # Nodos tienen lat/lon directo; ways/relations usan el centro calculado ('center')
        if tipo == "node":
            lon, lat = elemento.get("lon"), elemento.get("lat")
        else:
            centro = elemento.get("center", {})
            lon, lat = centro.get("lon"), centro.get("lat")
 
        if lon is None or lat is None:
            continue
 
        tags = elemento.get("tags", {})
 
        feature = {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [lon, lat],
            },
            "properties": {
                "osm_id": elemento.get("id"),
                "osm_type": tipo,
                "amenity": tags.get("amenity"),
                "name": tags.get("name"),
                "capacity": tags.get("capacity"),
                "bicycle_parking": tags.get("bicycle_parking"),  # tipo de aparcabicis
                "parking": tags.get("parking"),  # tipo de aparcamiento (surface, underground...)
                "fee": tags.get("fee"),
                "access": tags.get("access"),
                "operator": tags.get("operator"),
                # Guardamos también todas las etiquetas originales por si acaso
                "tags_completos": tags,
            },
        }
        features.append(feature)
 
    return {
        "type": "FeatureCollection",
        "features": features,
    }

In [ ]:
query = construir_query(CIUDAD, AMENITIES)
data = descargar_datos(query)

n_elementos = len(data.get("elements", []))
print(f"Elementos descargados: {n_elementos}")

geojson = a_geojson(data)

with open(SALIDA_GEOJSON, "w", encoding="utf-8") as f:
    json.dump(geojson, f, ensure_ascii=False, indent=2)

print(f"Guardado en: {SALIDA_GEOJSON}")

# Resumen rápido por tipo de amenity
resumen = {}
for feat in geojson["features"]:
    a = feat["properties"]["amenity"]
    resumen[a] = resumen.get(a, 0) + 1
print("Resumen por tipo de amenity:")
for k, v in resumen.items():
    print(f"  {k}: {v}")

Consultando https://overpass-api.de/api/interpreter (intento 1/3) ...
  Fallo con https://overpass-api.de/api/interpreter: 406 Client Error: Not Acceptable for url: https://overpass-api.de/api/interpreter. Esperando 15s antes de reintentar...
Consultando https://overpass-api.de/api/interpreter (intento 2/3) ...
  Fallo con https://overpass-api.de/api/interpreter: 406 Client Error: Not Acceptable for url: https://overpass-api.de/api/interpreter. Esperando 30s antes de reintentar...
Consultando https://overpass-api.de/api/interpreter (intento 3/3) ...
  Fallo con https://overpass-api.de/api/interpreter: 406 Client Error: Not Acceptable for url: https://overpass-api.de/api/interpreter. Esperando 60s antes de reintentar...
Se agotaron los reintentos con https://overpass-api.de/api/interpreter, probando el siguiente endpoint...

Consultando https://overpass.kumi.systems/api/interpreter (intento 1/3) ...
  429 Too Many Requests. Esperando 30s antes de reintentar...
Consultando https://overpa